# Pipeline Skripsi RFM — K-Means vs Hierarchical

Notebook ini menjalankan seluruh tahapan dari eksplorasi dataset sampai evaluasi dua algoritma. Dataset CSV tidak disimpan di GitHub; notebook akan mencari file di folder Colab atau cache KaggleHub.

In [ ]:
from pathlib import Path
import os, subprocess, sys, shutil

REPO_URL = 'https://github.com/rahmawnj/skripsi-RFM-Algoritma-K-Means-and-hirarical.git'
REPO_DIR = Path('/content/skripsi-RFM-Algoritma-K-Means-and-hirarical')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Folder kerja:', Path.cwd())
print('File Python:', len(list(REPO_DIR.glob('*.py'))))

In [ ]:
!pip -q install -r requirements.txt
print('Library siap digunakan.')

In [ ]:
DATA_NAME = 'Electronic_sales_Sep2023-Sep2024.csv'
candidates = [
    REPO_DIR / DATA_NAME,
    Path('/content') / DATA_NAME,
]
cache_root = Path.home() / '.cache/kagglehub/datasets/cameronseamons/electronic-sales-sep2023-sep2024/versions'
if cache_root.exists():
    candidates.extend(cache_root.glob(f'*/{DATA_NAME}'))

dataset_path = next((p for p in candidates if p.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError(
        'CSV belum ditemukan. Upload/copy Electronic_sales_Sep2023-Sep2024.csv '
        'ke panel Files Colab atau download dahulu menggunakan KaggleHub.'
    )

if dataset_path.resolve() != (REPO_DIR / DATA_NAME).resolve():
    shutil.copy2(dataset_path, REPO_DIR / DATA_NAME)
    dataset_path = REPO_DIR / DATA_NAME

print('Dataset siap:', dataset_path)
print('Ukuran file (MB):', round(dataset_path.stat().st_size / (1024**2), 2))
print('Catatan: CSV hanya disimpan di runtime Colab dan tidak otomatis di-push ke GitHub.')

## Jalankan seluruh tahapan

Jalankan cell berikut sampai selesai. Jika salah satu tahap error, hentikan dan periksa pesan error sebelum melanjutkan.

In [ ]:
scripts = [
    '01_eksplorasi_dataset.py',
    '02_perhitungan_rfm.py',
    '03_preprocessing.py',
    '04_kmeans.py',
    '05_hierarchical.py',
    '06_evaluasi_perbandingan.py',
]

for script in scripts:
    print('\n' + '=' * 85)
    print('MENJALANKAN:', script)
    print('=' * 85)
    subprocess.run([sys.executable, script], cwd=REPO_DIR, check=True)

print('\nSEMUA TAHAP SELESAI.')

## Ringkasan hasil dan visualisasi

Bagian ini menampilkan kembali tabel hasil dan grafik PNG yang dibuat oleh skrip.

In [ ]:
import pandas as pd
from IPython.display import display, Image

for filename in ['audit_kualitas_data.csv', 'distribusi_order_status.csv', 'hasil_rfm.csv']:
    path = REPO_DIR / filename
    if path.exists():
        print('\n', filename)
        display(pd.read_csv(path).head(10))

for filename in ['kmeans_metrics.csv', 'hierarchical_metrics.csv',
                 'perbandingan_algoritma_semua_k.csv', 'perbandingan_k_terbaik.csv',
                 'profil_cluster_kmeans.csv', 'profil_cluster_hierarchical.csv']:
    path = REPO_DIR / filename
    if path.exists():
        print('\n', filename)
        display(pd.read_csv(path))

images = [
    'visualisasi_distribusi_rfm.png', 'visualisasi_preprocessing.png',
    'grafik_elbow_kmeans.png', 'grafik_silhouette_kmeans.png',
    'dendrogram_hierarchical.png', 'grafik_silhouette_perbandingan.png',
    'grafik_dbi_perbandingan.png', 'grafik_waktu_perbandingan.png',
    'scatter_kmeans.png', 'scatter_hierarchical.png', 'grafik_ukuran_cluster.png',
]
for filename in images:
    path = REPO_DIR / filename
    if path.exists():
        print('\n' + filename)
        display(Image(filename=str(path)))

report = REPO_DIR / 'ringkasan_perbandingan.txt'
if report.exists():
    print('\nRINGKASAN PERBANDINGAN')
    print(report.read_text(encoding='utf-8'))

In [ ]:
# Tombol untuk mengunduh seluruh grafik PNG dalam satu file ZIP
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import display
import ipywidgets as widgets

png_files = sorted(REPO_DIR.glob('*.png'))
zip_path = REPO_DIR / 'hasil_visualisasi_rfm.zip'

if not png_files:
    print('Belum ada gambar PNG. Jalankan pipeline sampai selesai terlebih dahulu.')
else:
    with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as zipf:
        for image_path in png_files:
            zipf.write(image_path, arcname=image_path.name)

    download_button = widgets.Button(
        description=f'Download semua grafik ({len(png_files)} gambar)',
        button_style='success',
        icon='download',
        tooltip='Unduh seluruh grafik PNG dalam satu file ZIP'
    )
    download_status = widgets.Output()

    def download_all_graphs(_):
        from google.colab import files
        with download_status:
            download_status.clear_output()
            print('Menyiapkan file ZIP...')
            files.download(str(zip_path))

    download_button.on_click(download_all_graphs)
    display(download_button, download_status)
    print('Grafik yang akan dimasukkan:', ', '.join(p.name for p in png_files))

## Catatan penting untuk skripsi

- Hasil GitHub Actions hanya menguji pipeline menggunakan data sintetis; angka penelitian harus berasal dari CSV asli.
- Periksa asumsi satu baris = satu pesanan dan definisi `Total Price` + `Add-on Total` sebelum menetapkan RFM final.
- Hierarchical Ward standar membutuhkan memori besar; skrip memiliki batas pengaman 12.000 pelanggan.
- Jangan mengubah atau menulis kesimpulan penelitian sebelum tabel, grafik, dan profil cluster dari dataset asli diamati.